# P8 – Theory and simulation

This notebook reproduces the analytical results and the Monte Carlo study of the methodological note
(`P8_theorie_et_modelisation.pdf`) with the package code. Nothing here uses real data.

1. Proposition 2: with two assets and sign restrictions only, HB is a transformation of the correlation.
2. Proposition 3: the bond beta is a variance-weighted average of shock-specific hedge ratios.
3. Monte Carlo: the real-time estimator recovers the true hedge-breaking share.
4. Proposition 4: P8-A differs from the volatility target only in high-volatility hedge months.

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from p8regimes.config import Params
from p8regimes.simulate import B0, true_hb
from p8regimes.structural import RotationBank, identified_set
from p8regimes.indicator import shock_stats, run_indicator, realized_stats
from p8regimes.pipeline import demo_panel
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

## 1. Proposition 2 – two assets, signs only

In [ ]:
def dim2_set(rho, n=200_000, seed=1):
    P = np.linalg.cholesky(np.array([[1.0, rho], [rho, 1.0]]))
    th = np.random.default_rng(seed).uniform(0, 2 * np.pi, n)
    c1, c2 = np.stack([np.cos(th), np.sin(th)]), np.stack([-np.sin(th), np.cos(th)])
    vals = []
    for h, k in ((P @ c1, P @ c2), (P @ c2, P @ c1)):
        ok = (np.sign(h[0]) == -np.sign(h[1])) & (np.sign(k[0]) == np.sign(k[1]))
        vals.append(k[0][ok] ** 2)
    return np.concatenate(vals)

rows = []
for rho in np.linspace(-0.9, 0.9, 19):
    v = dim2_set(rho)
    rows.append({"rho": rho, "min": v.min(), "median": np.median(v), "max": v.max(),
                 "theory min": rho**2 if rho >= 0 else 0.0, "theory median": (1 + rho) / 2,
                 "theory max": 1.0 if rho >= 0 else 1 - rho**2})
t2 = pd.DataFrame(rows).set_index("rho"); t2

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.fill_between(t2.index, 100 * t2["min"], 100 * t2["max"], color="#2a78d6", alpha=0.15, label="identified set")
ax.plot(t2.index, 100 * t2["median"], color="#2a78d6", label="median under uniform rotations")
ax.axhline(50, color="#52514e", lw=0.8, ls=":"); ax.set_xlabel("stock-bond correlation"); ax.set_ylabel("HB (%)")
ax.legend(loc="upper left", frameon=False);

## 2. Proposition 3 – the bond beta

In [ ]:
rng = np.random.default_rng(3)
u = (rng.standard_normal((500, 4)) * [1.0, 1.5, 0.8, 1.2]) @ B0.T
u -= u.mean(axis=0)
B_all, rate, used = identified_set(np.cov(u.T), RotationBank(seed=2), n_accept=500)
st = shock_stats(B_all, u, duration=7.0)
rb, re = -7 * (0.6 * u[:, 1] + 0.4 * u[:, 2]) / 100, u[:, 3]
print(f"OLS beta of the bond pocket on equities: {np.cov(rb, re)[0, 1] / re.var(ddof=1):.6f}")
print(f"sum_k s_k h_k over 500 admissible rotations: min {st['beta'].min():.6f}, max {st['beta'].max():.6f}")
print(f"HB over the same rotations: median {np.median(st['hb']):.3f}, 16-84 % band "
      f"{np.percentile(st['hb'], 16):.3f}-{np.percentile(st['hb'], 84):.3f} (acceptance {rate:.1%})")

## 3. Monte Carlo – 30 simulated years (about one minute)

In [ ]:
P = Params(n_accept=1000)
panel, sig = demo_panel(P, years=30, seed=2026)
ind = run_indicator(panel, P, use_cache=False, progress=False)
truth = true_hb(sig.to_numpy(), panel.daily.index, ind.index)
rs = realized_stats(panel.daily, ind.index)
stats = pd.Series({
    "corr(estimated, true)": ind["hb"].corr(truth),
    "mean bias": (ind["hb"] - truth).mean(),
    "RMSE": np.sqrt(((ind["hb"] - truth) ** 2).mean()),
    "band 16-84 coverage": ((truth >= ind["hb_p16"]) & (truth <= ind["hb_p84"])).mean(),
    "regime accuracy, HB >= 50 %": ((ind["hb"] >= 0.5) == (truth >= 0.5)).mean(),
    "regime accuracy, rolling corr > 0": ((rs["corr"] > 0) == (truth >= 0.5)).mean(),
})
stats.to_frame("value")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.fill_between(ind.index, 100 * ind["hb_p16"], 100 * ind["hb_p84"], color="#2a78d6", alpha=0.15, lw=0)
ax.plot(ind.index, 100 * ind["hb"], color="#2a78d6", label="estimated (median)")
ax.plot(truth.index, 100 * truth, color="#0b0b0b", lw=1, label="true share")
ax.axhline(50, color="#52514e", lw=0.8, ls=":"); ax.set_ylim(0, 100); ax.set_ylabel("hedge-breaking share (%)")
ax.legend(loc="upper right", frameon=False);

**Reading.** The estimator tracks the truth closely, but to classify the *current* regime the sign of the
rolling correlation does as well: with similar hedge ratios in the two families, HB ≥ 50 % and a positive
correlation are almost the same statement (proposition 3). The value of the decomposition has to come from
forecasting (shock-specific persistence, P8-beta) and from explaining the regime.

## 4. Proposition 4 – P8-A versus the volatility target

In [ ]:
from p8regimes.backtest import Strategy, run_backtest
idx = pd.date_range("2000-01-31", periods=121, freq="ME"); rng = np.random.default_rng(4)
mon = pd.DataFrame({"r_e": rng.normal(0.006, 0.045, 121), "r_b": rng.normal(0.003, 0.02, 121), "r_f": 0.002}, index=idx)
sig = pd.DataFrame({"C": rng.integers(0, 2, 121).astype(float)}, index=idx)
risk = pd.DataFrame({"sigma_ref": 0.10, "sigma_6040": rng.uniform(0.07, 0.16, 121), "sigma_e": 0.2}, index=idx)
risk["k"] = np.minimum(1, risk["sigma_ref"] / risk["sigma_6040"])
res = run_backtest(mon, sig, risk, [Strategy("60/40", None, "A", ""), Strategy("VC", None, "VT", ""),
                                     Strategy("P8", "C", "A", "")], Params(cost_bp=0.0))
lhs = res["P8"]["x"] - res["VC"]["x"]
rhs = (1 - res["P8"]["C"]) * (1 - res["P8"]["k"]) * (res["60/40"]["R"] - res["60/40"]["R_f"])
print("max |difference| between the two sides of proposition 4:", float((lhs - rhs).abs().max()))